# Ensemble Affinity Analysis (On-the-fly)

This notebook computes PRP and RA summaries in memory from merged benchmark files.

No summary CSVs are created and figures are shown inline only.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

In [ ]:
# Set run_id explicitly, or keep None to use latest run folder.
run_id = None

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
bench_root = repo_root / "data" / "results" / "affinity_benchmark"

if run_id is None:
    run_dirs = sorted([p for p in bench_root.iterdir() if p.is_dir()])
    if not run_dirs:
        raise FileNotFoundError(f"No benchmark runs found under {bench_root}")
    run_dir = run_dirs[-1]
else:
    run_dir = bench_root / run_id

prp_path = run_dir / "merged_prp_affinities.csv"
ra_path = run_dir / "merged_ra_affinities.csv"

if not prp_path.exists() or not ra_path.exists():
    raise FileNotFoundError(
        "Merged files not found. Run scripts/run_postprocessing.py first for this run_id."
    )

prp_df = pd.read_csv(prp_path)
ra_df = pd.read_csv(ra_path)

print("Run directory:", run_dir)
print("PRP rows:", len(prp_df), "| Models:", prp_df['Model_Slug'].nunique())
print("RA rows:", len(ra_df), "| Models:", ra_df['Model_Slug'].nunique())

In [ ]:
# Quick coverage checks
display(prp_df[['Model_Slug', 'Abstract_Index']].drop_duplicates().groupby('Model_Slug').size().rename('n_abstracts').reset_index())
display(ra_df[['Model_Slug', 'Abstract_Index']].drop_duplicates().groupby('Model_Slug').size().rename('n_abstracts').reset_index())

# PRP Affinity level
Affinity level of an **abstract** to each **primary research programme** (PRP)

In [ ]:
fig = px.box(
    prp_df,
    x='Abstract_Index',
    y='LLM_Affinity',
    # color='Model_Slug',
    points='all',
    # hover_data='PRP_Name',
    # facet_col='PRP_Name',
    color = 'PRP_Name',
    title=f'PRP Affinity Distribution by Abstract'
 )
fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
fig = px.box(
    prp_df,
    color='Model_Slug',
    y='LLM_Affinity',
    x = 'PRP_Name',
    title='Affinity Distribution by PRP across models',
 )
fig.update_layout(xaxis_title='PRP', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

## PRP Summaries

In [ ]:
prp_summary = (
    prp_df.groupby(['Model_Slug', 'PRP_Name'], as_index=False)['LLM_Affinity']
    .agg(mean_affinity='mean', std_affinity='std', n='count')
)
display(prp_summary.head(20))

In [ ]:
# 
fig = px.scatter(
    prp_summary,
    color='PRP_Name',
    y='mean_affinity',
    x = 'Model_Slug',
    title='PRP Affinity Distribution by Model',
    # points = 'all',
 )
fig.update_layout(xaxis_title='PRP', yaxis_title='LLM Affinity')
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
prp_heat = prp_summary.pivot(index='Model_Slug', columns='PRP_Name', values='mean_affinity').sort_index()
fig = px.imshow(
    prp_heat,
    text_auto='.1f',
    color_continuous_scale='Viridis',
    title='Mean PRP Affinity (Model x PRP)',
    aspect='auto'
 )
fig.update_layout(xaxis_title='PRP Name', yaxis_title='Model')
fig.show()

In [ ]:
# Top-1 PRP agreement per abstract
prp_top1 = (
    prp_df.sort_values(['Model_Slug', 'Abstract_Index', 'LLM_Affinity'], ascending=[True, True, False])
    .groupby(['Model_Slug', 'Abstract_Index'], as_index=False)
    .first()[['Model_Slug', 'Abstract_Index', 'PRP_Name','LLM_Affinity']]
)

prp_agreement = (
    prp_top1.groupby('Abstract_Index')['PRP_Name']
    .nunique()
    .rename('n_unique_top1_prp')
    .reset_index()
)

print('Full agreement rate (all models same top-1 PRP):', (prp_agreement['n_unique_top1_prp'] == 1).mean())
display(prp_agreement['n_unique_top1_prp'].value_counts().sort_index().rename('n_abstracts'))

## RA Summaries

In [ ]:
ra_df.head(20)

In [ ]:
# fig = px.box(
#     ra_df,
#     x='Abstract_Index',
#     y='LLM_Affinity',
#     color='Model_Slug',
#     points='all',
#     # facet_row='PRP_Name',
#     # color = 'PRP_Name',
#     title=f'RA Affinity Distribution by Abstract'
#  )
# fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
# # fig.update_xaxes(tickangle=45)
# fig.show()

In [ ]:
fig = px.scatter(
    ra_df[ra_df['Abstract_Index'].isin([7,8,9])],
    # ra_df,
    x='RA2025_ID',
    y='LLM_Affinity',
    color='Model_Slug',
    facet_col = 'Abstract_Index',
    # points='all',
    hover_data = 'Model_Slug',
    # facet_row='PRP_Name',
    # color = 'PRP_Name',
    # title=f'RA Affinity Distribution by Abstract'
 )
fig.update_layout(xaxis_title='RA question', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
ra_summary = (
    ra_df.groupby(['Model_Slug', 'RA2025_ID'], as_index=False)['LLM_Affinity']
    .agg(mean_affinity='mean', std_affinity='std', n='count')
)
display(ra_summary.head(20))

In [ ]:
fig = px.box(
    ra_df,
    x='Model_Slug',
    y='LLM_Affinity',
    title='RA Affinity Distribution by Model'
 )
fig.update_layout(xaxis_title='Model', yaxis_title='LLM Affinity')
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# Top-1 RA agreement per abstract
ra_top1 = (
    ra_df.sort_values(['Model_Slug', 'Abstract_Index', 'LLM_Affinity'], ascending=[True, True, False])
    .groupby(['Model_Slug', 'Abstract_Index'], as_index=False)
    .first()[['Model_Slug', 'Abstract_Index', 'RA2025_ID']]
)

ra_agreement = (
    ra_top1.groupby('Abstract_Index')['RA2025_ID']
    .nunique()
    .rename('n_unique_top1_ra')
    .reset_index()
)

print('Full agreement rate (all models same top-1 RA):', (ra_agreement['n_unique_top1_ra'] == 1).mean())
display(ra_agreement['n_unique_top1_ra'].value_counts().sort_index().rename('n_abstracts'))

### RA Abstract-wise Affinity Distribution (Selected Abstract Index)
Using the same `Abstract_Index`, inspect how RA affinities vary across models for that abstract.

In [ ]:
target_abstract_index=8
ra_abstract_summary = ra_df.loc[ra_df['Abstract_Index'] == target_abstract_index].copy()
if ra_abstract_summary.empty:
    raise ValueError(f"Abstract_Index {target_abstract_index} not found in RA data")

print(f"RA abstract-wise view for Abstract_Index={target_abstract_index}")
print("Rows:", len(ra_abstract_summary), "| Models:", ra_abstract_summary['Model_Slug'].nunique(), "| RAs:", ra_abstract_summary['RA2025_ID'].nunique())

display(
    ra_abstract_summary.groupby('Model_Slug', as_index=False)['LLM_Affinity']
    .agg(mean_affinity='mean', median_affinity='median', std_affinity='std', min_affinity='min', max_affinity='max', n='count')
    .sort_values('mean_affinity', ascending=False)
)

fig = px.box(
    ra_abstract_summary,
    x='Model_Slug',
    y='LLM_Affinity',
    color='RA2025_ID',
    points='all',
    title=f'RA Affinity Distribution by Model (Abstract_Index={target_abstract_index})'
 )
fig.update_layout(xaxis_title='Model', yaxis_title='LLM Affinity')
fig.update_xaxes(tickangle=45)
fig.show()

## Coherency Analysis: Model Agreement on Affinity Levels

Analyze whether all models assign similar affinity scores to the same questions/abstracts.

In [ ]:
# PRP Coherency: Measure agreement (CV) across models per abstract-PRP pair
prp_coherency = (
    prp_df.groupby(['Abstract_Index', 'PRP_Name'], as_index=False)['LLM_Affinity']
    .agg(
        mean_affinity='mean',
        std_affinity='std',
        min_affinity='min',
        max_affinity='max',
        n_models='count'
    )
)

# Coefficient of Variation (CV) as coherency metric: lower CV = higher agreement
prp_coherency['cv'] = prp_coherency['std_affinity'] / (prp_coherency['mean_affinity'] + 1e-8)
prp_coherency['range'] = prp_coherency['max_affinity'] - prp_coherency['min_affinity']

print("PRP Coherency Summary:")
print(f"Mean CV across all abstract-PRP pairs: {prp_coherency['cv'].mean():.3f}")
print(f"Abstracts with perfect agreement (CV=0): {(prp_coherency['cv'] == 0).sum()}")
print("\nBottom 10 coherent (high CV) abstract-PRP pairs:")
display(prp_coherency.nlargest(10, 'cv')[['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'range']])
print("\nTop 10 coherent (low CV) abstract-PRP pairs:")
display(prp_coherency.nsmallest(10, 'cv')[['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'range']])

In [ ]:
# RA Coherency: Measure agreement (CV) across models per abstract-RA pair
ra_coherency = (
    ra_df.groupby(['Abstract_Index', 'RA2025_ID'], as_index=False)['LLM_Affinity']
    .agg(
        mean_affinity='mean',
        std_affinity='std',
        min_affinity='min',
        max_affinity='max',
        n_models='count'
    )
)

ra_coherency['cv'] = ra_coherency['std_affinity'] / (ra_coherency['mean_affinity'] + 1e-8)
ra_coherency['range'] = ra_coherency['max_affinity'] - ra_coherency['min_affinity']

print("RA Coherency Summary:")
print(f"Mean CV across all abstract-RA pairs: {ra_coherency['cv'].mean():.3f}")
print(f"Abstracts with perfect agreement (CV=0): {(ra_coherency['cv'] == 0).sum()}")
print("\nBottom 10 coherent (high CV) abstract-RA pairs:")
display(ra_coherency.nlargest(10, 'cv')[['Abstract_Index', 'RA2025_ID', 'mean_affinity', 'cv', 'range']])
print("\nTop 10 coherent (low CV) abstract-RA pairs:")
display(ra_coherency.nsmallest(10, 'cv')[['Abstract_Index', 'RA2025_ID', 'mean_affinity', 'cv', 'range']])

In [ ]:
# Abstract-level coherency: aggregate CV per abstract (for both PRP and RA)
prp_abstract_coherency = (
    prp_coherency.groupby('Abstract_Index', as_index=False)['cv']
    .agg(mean_cv='mean', max_cv='max', min_cv='min', std_cv='std', n_pairs='count')
    .sort_values('mean_cv', ascending=True)
)

ra_abstract_coherency = (
    ra_coherency.groupby('Abstract_Index', as_index=False)['cv']
    .agg(mean_cv='mean', max_cv='max', min_cv='min', std_cv='std', n_pairs='count')
    .sort_values('mean_cv', ascending=True)
)

print("PRP Abstract-level Coherency (Average CV per Abstract):")
print(f"Most coherent abstracts (lowest mean CV):")
display(prp_abstract_coherency.head(10))
print(f"\nLeast coherent abstracts (highest mean CV):")
display(prp_abstract_coherency.tail(10))

print("\n" + "="*60)
print("\nRA Abstract-level Coherency (Average CV per Abstract):")
print(f"Most coherent abstracts (lowest mean CV):")
display(ra_abstract_coherency.head(10))
print(f"\nLeast coherent abstracts (highest mean CV):")
display(ra_abstract_coherency.tail(10))

In [ ]:
# Visualization: Coherency Distribution
fig = px.histogram(
    prp_coherency[prp_coherency['cv'].notna()],
    x='cv',
    nbins=30,
    title='PRP Coherency Distribution (Coefficient of Variation)',
    labels={'cv': 'Coefficient of Variation', 'count': 'Number of Pairs'}
)
fig.add_vline(x=prp_coherency['cv'].mean(), line_dash="dash", line_color="red", 
              annotation_text=f"Mean CV: {prp_coherency['cv'].mean():.3f}")
fig.show()

fig = px.histogram(
    ra_coherency[ra_coherency['cv'].notna()],
    x='cv',
    nbins=30,
    title='RA Coherency Distribution (Coefficient of Variation)',
    labels={'cv': 'Coefficient of Variation', 'count': 'Number of Pairs'}
)
fig.add_vline(x=ra_coherency['cv'].mean(), line_dash="dash", line_color="red",
              annotation_text=f"Mean CV: {ra_coherency['cv'].mean():.3f}")
fig.show()

In [ ]:
# Abstract-level coherency scatter plots
fig = px.scatter(
    prp_abstract_coherency,
    x='Abstract_Index',
    y='mean_cv',
    size='n_pairs',
    color='mean_cv',
    hover_data=['max_cv', 'std_cv'],
    title='PRP Abstract-level Coherency (Lower CV = Higher Agreement)',
    labels={'mean_cv': 'Mean CV (Coherency)', 'Abstract_Index': 'Abstract Index'},
    color_continuous_scale='RdYlGn_r'
)
fig.show()

fig = px.scatter(
    ra_abstract_coherency,
    x='Abstract_Index',
    y='mean_cv',
    size='n_pairs',
    color='mean_cv',
    hover_data=['max_cv', 'std_cv'],
    title='RA Abstract-level Coherency (Lower CV = Higher Agreement)',
    labels={'mean_cv': 'Mean CV (Coherency)', 'Abstract_Index': 'Abstract Index'},
    color_continuous_scale='RdYlGn_r'
)
fig.show()

In [ ]:
# Pairwise Model Coherency: Correlation between model pairs per abstract-PRP
# This shows which model pairs agree most on affinity levels

from itertools import combinations

# For PRP: compute correlation between each pair of models per abstract
prp_model_pairs = []
for abstract_idx in prp_df['Abstract_Index'].unique():
    for prp_name in prp_df[prp_df['Abstract_Index'] == abstract_idx]['PRP_Name'].unique():
        subset = prp_df[(prp_df['Abstract_Index'] == abstract_idx) & (prp_df['PRP_Name'] == prp_name)]
        
        if len(subset) >= 2:
            models = subset['Model_Slug'].unique()
            for model1, model2 in combinations(models, 2):
                val1 = subset[subset['Model_Slug'] == model1]['LLM_Affinity'].values
                val2 = subset[subset['Model_Slug'] == model2]['LLM_Affinity'].values
                
                if len(val1) > 0 and len(val2) > 0:
                    diff = abs(val1[0] - val2[0])
                    prp_model_pairs.append({
                        'Abstract_Index': abstract_idx,
                        'PRP_Name': prp_name,
                        'Model_1': min(model1, model2),
                        'Model_2': max(model1, model2),
                        'Affinity_Difference': diff
                    })

prp_pairwise = pd.DataFrame(prp_model_pairs)

if len(prp_pairwise) > 0:
    print("PRP Model Pair Agreement Analysis:")
    print("\nModel pairs with highest average disagreement (highest mean affinity difference):")
    pair_stats = prp_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', std_diff='std', n_pairs='count'
    ).sort_values('mean_diff', ascending=False)
    display(pair_stats.head(10))
    
    print("\nModel pairs with lowest average disagreement (highest mean agreement):")
    display(pair_stats.sort_values('mean_diff', ascending=True).head(10))

In [ ]:
# For RA: compute correlation between each pair of models per abstract
ra_model_pairs = []
for abstract_idx in ra_df['Abstract_Index'].unique():
    for ra_id in ra_df[ra_df['Abstract_Index'] == abstract_idx]['RA2025_ID'].unique():
        subset = ra_df[(ra_df['Abstract_Index'] == abstract_idx) & (ra_df['RA2025_ID'] == ra_id)]
        
        if len(subset) >= 2:
            models = subset['Model_Slug'].unique()
            for model1, model2 in combinations(models, 2):
                val1 = subset[subset['Model_Slug'] == model1]['LLM_Affinity'].values
                val2 = subset[subset['Model_Slug'] == model2]['LLM_Affinity'].values
                
                if len(val1) > 0 and len(val2) > 0:
                    diff = abs(val1[0] - val2[0])
                    ra_model_pairs.append({
                        'Abstract_Index': abstract_idx,
                        'RA2025_ID': ra_id,
                        'Model_1': min(model1, model2),
                        'Model_2': max(model1, model2),
                        'Affinity_Difference': diff
                    })

ra_pairwise = pd.DataFrame(ra_model_pairs)

if len(ra_pairwise) > 0:
    print("RA Model Pair Agreement Analysis:")
    print("\nModel pairs with highest average disagreement (highest mean affinity difference):")
    pair_stats_ra = ra_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', std_diff='std', n_pairs='count'
    ).sort_values('mean_diff', ascending=False)
    display(pair_stats_ra.head(10))
    
    print("\nModel pairs with lowest average disagreement (highest mean agreement):")
    display(pair_stats_ra.sort_values('mean_diff', ascending=True).head(10))

In [ ]:
# Model Agreement Heatmaps: Show which models have smallest disagreements
if len(prp_pairwise) > 0:
    pair_stats = prp_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', n_pairs='count'
    ).sort_values('mean_diff', ascending=True)
    
    # Create a symmetrical matrix for heatmap
    all_models = sorted(set(list(pair_stats['Model_1'].unique()) + list(pair_stats['Model_2'].unique())))
    model_agreement_matrix = pd.DataFrame(0.0, index=all_models, columns=all_models)
    
    for _, row in pair_stats.iterrows():
        model_agreement_matrix.loc[row['Model_1'], row['Model_2']] = row['mean_diff']
        model_agreement_matrix.loc[row['Model_2'], row['Model_1']] = row['mean_diff']
    
    fig = px.imshow(
        model_agreement_matrix,
        text_auto='.2f',
        color_continuous_scale='RdYlGn_r',
        title='PRP Model Pairwise Disagreement (Lower = Better Agreement)',
        labels={'color': 'Mean Affinity Difference'},
        aspect='auto'
    )
    fig.update_layout(xaxis_title='Model', yaxis_title='Model')
    fig.show()

if len(ra_pairwise) > 0:
    pair_stats_ra = ra_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', n_pairs='count'
    ).sort_values('mean_diff', ascending=True)
    
    all_models_ra = sorted(set(list(pair_stats_ra['Model_1'].unique()) + list(pair_stats_ra['Model_2'].unique())))
    model_agreement_matrix_ra = pd.DataFrame(0.0, index=all_models_ra, columns=all_models_ra)
    
    for _, row in pair_stats_ra.iterrows():
        model_agreement_matrix_ra.loc[row['Model_1'], row['Model_2']] = row['mean_diff']
        model_agreement_matrix_ra.loc[row['Model_2'], row['Model_1']] = row['mean_diff']
    
    fig = px.imshow(
        model_agreement_matrix_ra,
        text_auto='.2f',
        color_continuous_scale='RdYlGn_r',
        title='RA Model Pairwise Disagreement (Lower = Better Agreement)',
        labels={'color': 'Mean Affinity Difference'},
        aspect='auto'
    )
    fig.update_layout(xaxis_title='Model', yaxis_title='Model')
    fig.show()

## Coherency Key Insights

Summary of model agreement levels on affinity assignments.

In [ ]:
# Summary statistics on model coherency
print("=" * 70)
print("COHERENCY ANALYSIS SUMMARY")
print("=" * 70)

print("\n📊 PRP COHERENCY:")
print(f"  • Mean CV (lower = more agreement): {prp_coherency['cv'].mean():.3f}")
print(f"  • Median CV: {prp_coherency['cv'].median():.3f}")
print(f"  • Perfect agreement cases (CV=0): {(prp_coherency['cv'] == 0).sum()} / {len(prp_coherency)}")
print(f"  • High disagreement cases (CV>1): {(prp_coherency['cv'] > 1).sum()} / {len(prp_coherency)}")
print(f"  • Range of affinities (mean): {prp_coherency['range'].mean():.3f}")
print(f"  • Most coherent abstract: {prp_abstract_coherency.iloc[0]['Abstract_Index']} (mean CV: {prp_abstract_coherency.iloc[0]['mean_cv']:.3f})")
print(f"  • Least coherent abstract: {prp_abstract_coherency.iloc[-1]['Abstract_Index']} (mean CV: {prp_abstract_coherency.iloc[-1]['mean_cv']:.3f})")

print("\n📊 RA COHERENCY:")
print(f"  • Mean CV (lower = more agreement): {ra_coherency['cv'].mean():.3f}")
print(f"  • Median CV: {ra_coherency['cv'].median():.3f}")
print(f"  • Perfect agreement cases (CV=0): {(ra_coherency['cv'] == 0).sum()} / {len(ra_coherency)}")
print(f"  • High disagreement cases (CV>1): {(ra_coherency['cv'] > 1).sum()} / {len(ra_coherency)}")
print(f"  • Range of affinities (mean): {ra_coherency['range'].mean():.3f}")
print(f"  • Most coherent abstract: {ra_abstract_coherency.iloc[0]['Abstract_Index']} (mean CV: {ra_abstract_coherency.iloc[0]['mean_cv']:.3f})")
print(f"  • Least coherent abstract: {ra_abstract_coherency.iloc[-1]['Abstract_Index']} (mean CV: {ra_abstract_coherency.iloc[-1]['mean_cv']:.3f})")

if len(prp_pairwise) > 0:
    pair_stats = prp_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].mean()
    print("\n🔗 PRP MODEL PAIRS - BEST AGREEMENT (Lowest Mean Difference):")
    for i, (_, row) in enumerate(pair_stats.sort_values('Affinity_Difference').head(3).iterrows(), 1):
        print(f"  {i}. {row['Model_1']} ↔ {row['Model_2']}: {row['Affinity_Difference']:.3f}")

if len(ra_pairwise) > 0:
    pair_stats_ra = ra_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].mean()
    print("\n🔗 RA MODEL PAIRS - BEST AGREEMENT (Lowest Mean Difference):")
    for i, (_, row) in enumerate(pair_stats_ra.sort_values('Affinity_Difference').head(3).iterrows(), 1):
        print(f"  {i}. {row['Model_1']} ↔ {row['Model_2']}: {row['Affinity_Difference']:.3f}")

print("\n" + "=" * 70)